# CVRP code walkthrough with RL4CO

We use the official [RL4CO repository](https://github.com/ai4co/rl4co) to inspect
the constructive decision process: environment state, feasibility mask, decoder
rollout, and POMO multi-start. The policy was pretrained on synthetic CVRP15
instances generated by RL4CO.


In [ ]:
# Run once if RL4CO is not installed in the active kernel:
# %pip install -q "rl4co==0.6.0"

from __future__ import annotations

from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd
import torch

try:
    from IPython.display import display
except ImportError:
    display = print
from rl4co.envs.routing import CVRPEnv, CVRPTWEnv
from rl4co.models import POMO
from rl4co.utils.decoding import process_logits
from rl4co.utils.ops import unbatchify

torch.manual_seed(1234)
torch.set_grad_enabled(False)

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 10,
    }
)

# Optional: set CVRP_EXPORT_DIR to save the figures used in the slides.
EXPORT_DIR = os.environ.get("CVRP_EXPORT_DIR")
if EXPORT_DIR:
    Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)


def finish_figure(fig: plt.Figure, name: str) -> None:
    fig.tight_layout()
    if EXPORT_DIR:
        fig.savefig(Path(EXPORT_DIR) / f"{name}.png", dpi=180, bbox_inches="tight")
    plt.show()


def instance_values(td):
    """Return CPU tensors for one reset RL4CO CVRP instance."""
    item = td[0].detach().cpu()
    locs = item["locs"]
    capacity = float(item["capacity"].squeeze())
    demands = torch.cat((torch.zeros(1), item["demand"] * capacity))
    return item, locs, demands, capacity


def plot_instance(td, ax=None, title="CVRP instance"):
    _, locs, demands, capacity = instance_values(td)
    if ax is None:
        _, ax = plt.subplots(figsize=(7.2, 5.4))

    ax.scatter(
        locs[1:, 0],
        locs[1:, 1],
        s=115,
        facecolor="#DDEEFF",
        edgecolor="#286090",
        linewidth=1.5,
    )
    ax.scatter(
        locs[0, 0],
        locs[0, 1],
        s=190,
        marker="s",
        facecolor="#F8D34F",
        edgecolor="#3A3A3A",
        linewidth=1.5,
    )
    for node, (x, y) in enumerate(locs):
        label = "D" if node == 0 else f"{node}\nq={int(demands[node])}"
        ax.text(x, y + 0.035, label, ha="center", va="bottom", fontsize=8)

    ax.set_title(f"{title}   (Q={capacity:.0f})", loc="left")
    ax.set(xlim=(0, 1), ylim=(0, 1), xticks=[], yticks=[])
    ax.set_aspect("equal")
    return ax


def split_routes(actions: torch.Tensor) -> list[list[int]]:
    sequence = [0, *actions.detach().cpu().tolist(), 0]
    routes, route = [], [0]
    for node in sequence[1:]:
        route.append(int(node))
        if node == 0:
            if len(route) > 2:
                routes.append(route)
            route = [0]
    if len(route) > 1:
        routes.append(route + [0])
    return routes


def plot_solution(td, actions, reward=None, ax=None, title="Constructed solution"):
    _, locs, demands, _ = instance_values(td)
    if ax is None:
        _, ax = plt.subplots(figsize=(7.2, 5.4))

    colors = ["#3D8DFF", "#F2994A", "#27AE60", "#9B51E0", "#EB5757"]
    for route_id, route in enumerate(split_routes(actions)):
        path = locs[route]
        ax.plot(
            path[:, 0],
            path[:, 1],
            marker="o",
            linewidth=2.1,
            markersize=4,
            color=colors[route_id % len(colors)],
            label=f"vehicle {route_id + 1}",
        )

    ax.scatter(
        locs[0, 0],
        locs[0, 1],
        s=190,
        marker="s",
        facecolor="#F8D34F",
        edgecolor="#3A3A3A",
        zorder=5,
    )
    for node, (x, y) in enumerate(locs):
        label = "D" if node == 0 else f"{node}\nq={int(demands[node])}"
        ax.text(x, y + 0.035, label, ha="center", va="bottom", fontsize=8)

    suffix = "" if reward is None else f" | length={-float(reward):.3f}"
    ax.set_title(f"{title}{suffix}", loc="left")
    ax.set(xlim=(0, 1), ylim=(0, 1), xticks=[], yticks=[])
    ax.set_aspect("equal")
    ax.legend(frameon=False, fontsize=8, loc="lower left")
    return ax


## 1. Instance and MDP environment

`CVRPEnv` stores static features and dynamic state in a `TensorDict`. A
transition is performed by writing an action and calling `env.step`.


In [ ]:
env = CVRPEnv(generator_params={"num_loc": 15})
td0 = env.reset(batch_size=[1])

_, locs, demands, capacity = instance_values(td0)
instance_table = pd.DataFrame(
    {
        "node": range(len(locs)),
        "type": ["depot", *(["customer"] * (len(locs) - 1))],
        "x": locs[:, 0].numpy().round(3),
        "y": locs[:, 1].numpy().round(3),
        "demand": demands.numpy().astype(int),
    }
)
display(instance_table)

fig, ax = plt.subplots(figsize=(7.2, 5.4))
plot_instance(td0, ax=ax)
finish_figure(fig, "01-rl4co-instance")


In [ ]:
first_action = int(torch.where(td0["action_mask"][0, 1:])[0][0] + 1)
td1 = td0.clone()
td1["action"] = torch.tensor([first_action])
td1 = env.step(td1)["next"]

state_table = pd.DataFrame(
    [
        {"state": "current_node", "before": 0, "after": int(td1["current_node"].item())},
        {
            "state": "used_capacity",
            "before": float(td0["used_capacity"].item()),
            "after": float(td1["used_capacity"].item()),
        },
        {
            "state": "visited_customers",
            "before": int(td0["visited"][..., 1:].sum()),
            "after": int(td1["visited"][..., 1:].sum()),
        },
    ]
)
display(state_table)


## 2. Feasibility mask

In RL4CO, `action_mask=True` means **feasible**. Visited customers and demands
exceeding the remaining capacity are removed before action selection.


In [ ]:
td_mask = td0.clone()
customer_demands = td_mask["demand"][0]
largest = int(customer_demands.argmax() + 1)
smallest = int(customer_demands.argmin() + 1)
remaining = customer_demands.clone()
remaining[[largest - 1, smallest - 1]] = -1
second_largest = int(remaining.argmax() + 1)
selected = [largest, smallest, second_largest]

for action in selected:
    td_mask["action"] = torch.tensor([action])
    td_mask = env.step(td_mask)["next"]

remaining_capacity = float(
    td_mask["vehicle_capacity"].item() - td_mask["used_capacity"].item()
)
feasible = td_mask["action_mask"][0].detach().cpu()
visited = td_mask["visited"][0].bool().detach().cpu()

mask_table = pd.DataFrame(
    {
        "node": range(len(locs)),
        "demand": demands.numpy().astype(int),
        "visited": visited.numpy(),
        "feasible": feasible.numpy(),
    }
)

print("Partial route:", [0, *selected])
print(f"Remaining normalized capacity: {remaining_capacity:.2f}")
display(mask_table)


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 5.4))

for node, (x, y) in enumerate(locs):
    is_depot = node == 0
    marker = "s" if is_depot else "o"
    facecolor = "#F8D34F" if is_depot else "#DDEEFF"
    alpha = 1.0 if feasible[node] or visited[node] else 0.28
    ax.scatter(
        x,
        y,
        s=175 if is_depot else 110,
        marker=marker,
        facecolor=facecolor,
        edgecolor="#3A3A3A",
        alpha=alpha,
    )
    if feasible[node]:
        ax.scatter(
            x,
            y,
            s=255 if is_depot else 180,
            marker=marker,
            facecolor="none",
            edgecolor="#238636",
            linewidth=2.2,
        )
    elif not visited[node]:
        ax.scatter(x, y, s=155, marker="x", color="#C13C37", linewidth=2.2)
    if node == int(td_mask["current_node"].item()):
        ax.scatter(
            x,
            y,
            s=300 if is_depot else 225,
            marker=marker,
            facecolor="none",
            edgecolor="#111827",
            linewidth=2.4,
        )

    label = "D" if is_depot else f"{node}\nq={int(demands[node])}"
    ax.text(x, y + 0.035, label, ha="center", va="bottom", fontsize=8)

ax.set_title(
    "RL4CO action mask\n"
    f"current={int(td_mask['current_node'].item())} | "
    f"remaining normalized capacity={remaining_capacity:.2f}",
    loc="left",
)
ax.text(
    0.01,
    0.01,
    "green ring = feasible   red x = capacity-infeasible   black ring = current",
    transform=ax.transAxes,
    fontsize=8,
    color="#4B5563",
)
ax.set(xlim=(0, 1), ylim=(0, 1), xticks=[], yticks=[])
ax.set_aspect("equal")
finish_figure(fig, "02-rl4co-mask")


## 3. Decoder and constructive rollout

The policy encodes the instance once, then repeats logits → mask → action →
environment step. We load a pretrained CVRP15 policy and expose one decoder
step before the complete rollout.


In [ ]:
pomo = POMO(
    env,
    policy_kwargs={"embed_dim": 128, "num_encoder_layers": 6},
    num_augment=1,
)
pomo.eval()
policy = pomo.policy

checkpoint_relative = Path(
    "outputs/checkpoints/pomo-cvrp15-standard/pomo_cvrp15_policy.pt"
)
checkpoint_candidates = [checkpoint_relative, Path("..") / checkpoint_relative]
checkpoint_path = next((path for path in checkpoint_candidates if path.exists()), None)
if checkpoint_path is None:
    raise FileNotFoundError(
        "Pretrained policy not found. Run notebooks/train_pomo_cvrp15.py first."
    )

policy.load_state_dict(
    torch.load(checkpoint_path, map_location="cpu", weights_only=True)
)
print(f"Loaded pretrained policy: {checkpoint_path.resolve()}")

hidden, _ = policy.encoder(td_mask)
_, _, cache = policy.decoder.pre_decoder_hook(td_mask, env, hidden, num_starts=0)
logits, decoder_mask = policy.decoder(td_mask, cache, num_starts=0)
logprobs = process_logits(
    logits.clone(),
    decoder_mask,
    temperature=policy.temperature,
    tanh_clipping=policy.tanh_clipping,
    mask_logits=policy.mask_logits,
)
probabilities = logprobs.exp()[0].detach().cpu()

decoder_table = pd.DataFrame(
    {
        "node": range(len(probabilities)),
        "feasible": decoder_mask[0].detach().cpu().numpy(),
        "probability": probabilities.numpy().round(4),
    }
).sort_values("probability", ascending=False)
display(decoder_table)

fig, ax = plt.subplots(figsize=(8.2, 4.5))
bar_colors = ["#3D8DFF" if feasible[i] else "#D8D8D8" for i in range(len(feasible))]
ax.bar(range(len(probabilities)), probabilities, color=bar_colors)
ax.set(
    xlabel="Next node",
    ylabel="Probability",
    title="One attention-decoder step after feasibility masking",
)
ax.set_xticks(range(len(probabilities)))
finish_figure(fig, "03-rl4co-decoder-step")


In [ ]:
greedy_out = policy(
    td0.clone(),
    env,
    phase="test",
    decode_type="greedy",
    return_actions=True,
)
greedy_actions = greedy_out["actions"][0]
greedy_reward = greedy_out["reward"][0]

env.check_solution_validity(td0, greedy_out["actions"])
print("Action sequence:", greedy_actions.tolist())
print(f"Route length: {-float(greedy_reward):.3f}")

fig, ax = plt.subplots(figsize=(7.2, 5.4))
plot_solution(
    td0,
    greedy_actions,
    reward=greedy_reward,
    ax=ax,
    title="Single-start greedy rollout",
)
finish_figure(fig, "04-rl4co-greedy-rollout")


## 4. POMO and constrained VRPs

POMO decodes one trajectory per first customer with the same policy. The
instance mean reward is the shared training baseline.


In [ ]:
num_starts = env.get_num_starts(td0)
multi_out = policy(
    td0.clone(),
    env,
    phase="test",
    decode_type="multistart_greedy",
    num_starts=num_starts,
    return_actions=True,
)

multi_rewards = unbatchify(multi_out["reward"], num_starts)
multi_actions = unbatchify(multi_out["actions"], num_starts)
advantages = multi_rewards - multi_rewards.mean(dim=-1, keepdim=True)
best_start_idx = int(multi_rewards[0].argmax())
best_actions = multi_actions[0, best_start_idx]
best_reward = multi_rewards[0, best_start_idx]

env.check_solution_validity(td0, best_actions[None])

pomo_table = pd.DataFrame(
    {
        "first_customer": range(1, num_starts + 1),
        "route_length": (-multi_rewards[0]).detach().cpu().numpy().round(3),
        "reward_advantage": advantages[0].detach().cpu().numpy().round(3),
    }
)
display(pomo_table.sort_values("route_length"))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.1))
bar_colors = [
    "#3D8DFF" if index == best_start_idx else "#B8DDF5"
    for index in range(num_starts)
]
axes[0].bar(pomo_table["first_customer"].astype(str), pomo_table["route_length"], color=bar_colors)
axes[0].axhline(
    -float(greedy_reward),
    color="#333333",
    linestyle="--",
    linewidth=1.5,
    label="single start",
)
axes[0].set(xlabel="First customer", ylabel="Route length", title="POMO multi-start candidates")
axes[0].legend(frameon=False)

plot_solution(
    td0,
    best_actions,
    reward=best_reward,
    ax=axes[1],
    title=f"Best start: customer {best_start_idx + 1}",
)
finish_figure(fig, "05-rl4co-pomo-multistart")


In [ ]:
tw_env = CVRPTWEnv(generator_params={"num_loc": 15})
tw_td = tw_env.reset(batch_size=[1])

extension_table = pd.DataFrame(
    [
        {
            "RL4CO hook": "state",
            "CVRP": "used_capacity",
            "CVRPTW example": "current_time, time_windows, durations",
        },
        {
            "RL4CO hook": "get_action_mask",
            "CVRP": "visited & capacity",
            "CVRPTW example": "CVRP mask & reachable before deadline",
        },
        {
            "RL4CO hook": "_step",
            "CVRP": "update load and visited set",
            "CVRPTW example": "also update travel/service time",
        },
    ]
)
display(extension_table)
print("Additional CVRPTW state keys:", sorted(set(tw_td.keys()) - set(td0.keys())))


## Takeaways

- RL4CO separates the environment transition from the neural policy.
- Hard constraints enter through dynamic state and a boolean feasibility mask.
- POMO keeps the same decoder and adds parallel starts plus a shared baseline.
- New routing constraints usually extend `state`, `get_action_mask`, and `_step`.
